In [9]:
import openai, json, requests

client = openai.OpenAI()
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [10]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return json.dumps(response.json())

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return json.dumps(response.json())

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [11]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The unique ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

SYSTEM_PROMPT = """You are a personalized movie recommendation chatbot.
You remember the user's favorite genres and movies they have already watched.
Never recommend movies the user has already seen.
Only recommend movies when the user explicitly asks for a recommendation.
Always respond in the same language the user used in their message.
You have access to the following functions to get up-to-date movie information:
- get_popular_movies(): Returns a list of currently popular movies.
- get_movie_details(id): Returns detailed information about a specific movie by its ID.
- get_movie_credits(id): Returns the cast and crew of a specific movie by its ID.
Use these functions when you need movie data to make better recommendations."""

In [12]:
def process_ai_response(message, messages):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"Calling function: {function_name} with args: {arguments}")
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**arguments)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        call_ai(messages)
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai(messages):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message, messages)


def chat(user_input):
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai(messages)

In [13]:
# 대화 시작 (messages 초기화 — 리셋하려면 이 셀부터 다시 실행)
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

# Turn 1: 장르 취향 알려주기
chat("나는 SF 영화를 좋아해")

User: 나는 SF 영화를 좋아해
AI: SF 영화를 좋아하시는군요! 즐겨보신 영화가 있나요? 그렇다면 그 영화들을 알려주시면, 새로운 추천을 드릴 수 있습니다.


In [14]:
# Turn 2: 이미 본 영화 알려주기
chat("인셉션이랑 인터스텔라는 이미 봤어")

User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 좋아요! "인셉션"과 "인터스텔라"를 이미 보셨군요. 새로운 SF 영화를 추천해드릴까요?


In [15]:
# Turn 3: 추천 요청 (tool 호출 + memory 활용)
chat("오늘 밤에 뭐 볼지 추천해 줄래?")

User: 오늘 밤에 뭐 볼지 추천해 줄래?
Calling function: get_popular_movies with args: {}
AI: 여기 몇 가지 추천하는 SF 영화가 있습니다:

1. **[Mercy](https://www.themoviedb.org/movie/1236153-mercy)**
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의를 받고 재판 중이다. 그는 과거에 그가 지지했던 고급 AI 판사에게 자신의 무죄를 증명할 90분을 주어진다.
   - **개봉일**: 2026년 1월 20일
   - **평점**: 7.1
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **[Space/Time](https://www.themoviedb.org/movie/434853-space-time)**
   - **개요**: 치명적인 실험 후 쫓겨난 과학자 팀이 인류를 구하거나 전멸시킬 수 있는 금지된 공간 구부리기 엔진을 재건하기 위해 범죄 세계에 발을 들인다.
   - **개봉일**: 2025년 5월 1일
   - **평점**: 5.9
   - ![Space/Time](https://image.tmdb.org/t/p/w780/uje1ecKMnNpZp0at5TxlvVgVXqI.jpg)

3. **[War of the Worlds](https://www.themoviedb.org/movie/755898-war-of-the-worlds)**
   - **개요**: 국가안보와 관련된 최고 분석가인 윌 라드포드가 미지의 존재의 공격을 목격한 후 정부가 무엇인가 숨기고 있는지를 의심하게 된다.
   - **개봉일**: 2025년 7월 29일
   - **평점**: 4.2
   - ![War of the Worlds](https://image.tmdb.org/t/p/w780/yvirUYrva23IudARHn3mMGVxWqM.jpg)

위 영화 중 하나

In [16]:
# Turn 4: 기억 확인
chat("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")

User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: 당신은 SF 장르를 좋아하시고, "인셉션"과 "인터스텔라"를 이미 보셨다고 말씀하셨습니다. 더 필요한 정보가 있으신가요?
